In [ ]:
import math
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

In [ ]:
from numba import jit
from numba import cuda

In [ ]:
cuda.is_available()

In [ ]:
@pandas_udf(returnType=DoubleType())
def lonToX(lon:pd.Series)->pd.Series:
    return 6378137.0 * np.radians(lon)

@pandas_udf(returnType=DoubleType())
def latToY(lat:pd.Series)->pd.Series:
    return 6378137.0 * np.log(np.tan((math.pi * 0.25) + (0.5 * np.radians(lat))))

In [ ]:
@jit(nopython=True, parallel=True)
def _haversine(lon1,lat1,lon2,lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return 6378137.0 * 2.0 * np.arcsin(np.sqrt(a))

In [ ]:
@pandas_udf("double")
def haversine(lon1:pd.Series,lat1:pd.Series,lon2:pd.Series,lat2:pd.Series)->pd.Series:
    # Convert pandas series to numpy array for numba
    return pd.Series(_haversine(lon1.values,lat1.values,lon2.values,lat2.values))

In [ ]:
spark\
    .range(10_000_000)\
    .selectExpr("id","-180.0+360.0*rand() lon1","-90.0+180.0*rand() lat1","-180.0+360.0*rand() lon2","-90.0+180.0*rand() lat2")\
    .withColumn("meters", haversine("lon1","lat1","lon2","lat2"))\
    .createOrReplaceTempView("v0")

In [ ]:
%%time

sql("""select avg(meters) avg_meters from v0""").explain()